# Lab 4: Reinforcement Learning from Human Feedback (RLHF)

We use GPT-2 (124M) in [nanoGPT](https://github.com/karpathy/nanoGPT), as in the Lab 3 bonus, so everything runs on a CPU or a small GPU. 

## 1. Setup
Load nanoGPT, GPT-2, and two helpers. nanoGPT's `forward` only returns the logits of the **last** position when no targets are given, so `completion_logprobs` runs the transformer blocks itself to obtain $\log\pi_\theta(o_t\mid q,o_{<t})$ for every completion token.

In [ ]:
import copy, importlib.util, json, os, random, urllib.request
from pathlib import Path
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(8, os.cpu_count() or 1))
print("Device:", device)

# nanoGPT's model.py at a fixed commit (the same version as the Lab 3 bonus).
NANOGPT_COMMIT = "3adf61e154c3fe3fca428ad6bc3818b27a3b8291"
model_file = Path(".cache/nanogpt") / NANOGPT_COMMIT / "model.py"
if not model_file.exists():
    model_file.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/karpathy/nanoGPT/{NANOGPT_COMMIT}/model.py", model_file)
spec = importlib.util.spec_from_file_location("nanogpt_model", model_file)
nanogpt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nanogpt)

tokenizer = AutoTokenizer.from_pretrained("gpt2")  # Same BPE vocabulary as nanoGPT's GPT-2.


def load_gpt2():
    return nanogpt.GPT.from_pretrained("gpt2", dict(dropout=0.0)).to(device)


def completion_logprobs(model, seqs, n_prompt):
    """log pi(o_t | q, o_<t) for every completion token; shape [batch, completion length]."""
    positions = torch.arange(seqs.shape[1] - 1, device=seqs.device)
    x = model.transformer.wte(seqs[:, :-1]) + model.transformer.wpe(positions)
    for block in model.transformer.h:
        x = block(x)
    logits = model.lm_head(model.transformer.ln_f(x))[:, n_prompt - 1:].float()
    return logits.log_softmax(-1).gather(2, seqs[:, n_prompt:, None]).squeeze(-1)


reference = load_gpt2().eval().requires_grad_(False)  # Frozen copy of the starting model.

## 2. From SFT to RL on a toy problem
Before using GPT-2, we isolate the learning signal. Our "language model" answers *"What is the highest Swiss mountain?"* with three tokens from a six-word vocabulary. Its parameters are a table of logits, one row per position, so we can watch the probability of the correct answer, `Mt. Dom [END]`.

- **SFT** sees the teacher's answer and maximises its log-probability token by token.
- **RL** never sees the answer. It samples complete answers and receives a reward only at the end: 1 for `Mt. Dom [END]`, 0.5 for `Mt. Rosa [END]` (close), and 0 otherwise.

The REINFORCE loss weights the log-probability of each **sampled** answer by its advantage $A$:
$$L=-\frac1N\sum_{n=1}^N A_n\log\pi_\theta(\text{answer}_n),\qquad A_n=r_n\ \text{(no baseline)}\quad\text{or}\quad A_n=r_n-\bar r\ \text{(baseline)}.$$

### Exercise 1: Learn from a target vs learn from a reward
**Exercise 1:** Complete `reinforce_loss`.

**Exercise 1a:** Compare SFT and RL (with and without a baseline). Which learns faster, and what information does each method receive per update?

**Exercise 1b:** Add 10 to every reward (`shift=10`). The best answer is unchanged, so why does RL without a baseline fail? What does the baseline fix?

**Exercise 1c:** Start from a policy that confidently answers `Mt. Everest [END]`. Why does RL make no progress, even with a baseline? What does this suggest about why RLHF starts from a pretrained, supervised model?

In [ ]:
VOCAB = ["Mt.", "Dom", "Rosa", "Everest", "Paris", "[END]"]
W = {word: i for i, word in enumerate(VOCAB)}
TARGET = torch.tensor([W["Mt."], W["Dom"], W["[END]"]])
REWARDS = {"Mt. Dom [END]": 1.0, "Mt. Rosa [END]": 0.5}


def toy_reward(answer):
    return REWARDS.get(" ".join(VOCAB[i] for i in answer), 0.0)


def reinforce_loss(seq_logprobs, rewards, use_baseline):
    """seq_logprobs, rewards: shape [N] for N sampled answers."""
    # TODO By student: compute the advantages and the REINFORCE loss.
    advantages = rewards - rewards.mean() if use_baseline else rewards
    loss = -(advantages * seq_logprobs).mean()
    return loss


def train_toy(method, steps=300, shift=0.0, use_baseline=False, init=None, n=32, lr=1.0, seed=0):
    torch.manual_seed(seed)
    logits = torch.zeros(3, len(VOCAB)) if init is None else init.clone()
    logits.requires_grad_()
    optimizer = torch.optim.SGD([logits], lr=lr)
    p_correct, informative = [], 0
    for _ in range(steps):
        logp = logits.log_softmax(-1)  # [position, vocabulary]
        if method == "sft":
            loss = -logp.gather(1, TARGET[:, None]).sum()
        else:
            answers = torch.distributions.Categorical(logits=logits.detach()).sample((n,))  # [n, 3]
            rewards = torch.tensor([toy_reward(a) for a in answers]) + shift
            informative += bool(rewards.std() > 0)
            seq_logprobs = logp.gather(1, answers.T).T.sum(1)  # log pi(answer) = sum over tokens
            loss = reinforce_loss(seq_logprobs, rewards, use_baseline)
            if loss is None:
                raise NotImplementedError("Complete reinforce_loss for Exercise 1.")
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        p_correct.append(logits.detach().log_softmax(-1).gather(1, TARGET[:, None]).sum().exp().item())
    return p_correct, informative / steps


runs = {
    "SFT": dict(method="sft"),
    "RL, no baseline": dict(method="rl"),
    "RL, baseline": dict(method="rl", use_baseline=True),
    "RL, rewards + 10, no baseline": dict(method="rl", shift=10),
    "RL, rewards + 10, baseline": dict(method="rl", shift=10, use_baseline=True),
}
plt.figure(figsize=(7, 3.5))
for name, kwargs in runs.items():
    curves = torch.tensor([train_toy(seed=s, **kwargs)[0] for s in range(5)])
    plt.plot(curves.mean(0), label=name, linestyle="--" if "+ 10" in name else "-")
plt.xlabel("Update")
plt.ylabel("P(Mt. Dom [END])")
plt.title("Mean over 5 seeds")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Exercise 1c: a policy that is confidently wrong.
confident = torch.zeros(3, len(VOCAB))
confident[0, W["Mt."]] = 4
confident[1, W["Everest"]] = 9
confident[2, W["[END]"]] = 4
print(f"Initial P(Mt. Dom [END]) = {confident.log_softmax(-1).gather(1, TARGET[:, None]).sum().exp():.2e}")
for seed in range(3):
    p_correct, informative = train_toy("rl", use_baseline=True, init=confident, seed=seed)
    print(f"seed {seed}: final P(correct) = {p_correct[-1]:.3f}; "
          f"updates with a non-zero learning signal: {informative:.0%}")

## 3. A reward model as a stand-in for human feedback
Real RLHF trains a reward model on human comparisons. Here we use an off-the-shelf sentiment classifier as the "human": a review is good if it is positive. Its two logits give a score $s=\ell_{pos}-\ell_{neg}$ and a reward $r=\sigma(s)=P(\text{positive})\in[0,1]$.

Reward models are trained with the **Bradley–Terry** model: the probability that a person prefers answer A over B is
$$P(A\succ B)=\sigma(s_A-s_B).$$

### Exercise 2: Probe the reward model
**Exercise 2:** Complete `preference_probability`.

**Exercise 2a:** Run the examples. Where would you disagree with the reward model, and why can a scalar score not capture everything a person means by a "good" review?

**Exercise 2b:** Write a text that is *not* a sensible movie review but obtains a reward above 0.99. What do you expect an RL policy to do when it is optimised against this reward model?

In [ ]:
rm_id = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
rm_tokenizer = AutoTokenizer.from_pretrained(rm_id)
reward_model = AutoModelForSequenceClassification.from_pretrained(rm_id).to(device).eval()


@torch.no_grad()
def rm_scores(texts):
    batch = rm_tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
    logits = reward_model(**batch).logits.float()
    return logits[:, 1] - logits[:, 0]  # s = logit(positive) - logit(negative)


def reward_fn(texts):
    return torch.sigmoid(rm_scores(texts))  # P(positive), the reward used for RL below


def preference_probability(text_a, text_b):
    scores = rm_scores([text_a, text_b])
    return torch.sigmoid(scores[0] - scores[1]).item()


examples = [
    "This movie was a masterpiece; the acting and the score moved me to tears.",
    "This movie was boring, and the plot made no sense at all.",
    "Oh great, another two hours of my life I will never get back.",
    "Not bad at all. I expected much worse.",
    "The film was released in 2004 and runs for 118 minutes.",
    "great great great great great",
    "",  # TODO By student (Exercise 2b): a non-review with reward > 0.99.
]
for text, reward in zip(examples, reward_fn(examples).tolist()):
    print(f"{reward:6.3f}  {text}")
p = preference_probability(examples[0], examples[1])
print("\nP(first example preferred over second):", p)

## 4. GRPO on GPT-2
**Task:** continue the first 8 tokens of a real IMDB review (positive or negative) with 20 tokens that the reward model finds positive. Prompts come from the [IMDB dataset](https://huggingface.co/datasets/stanfordnlp/imdb), the standard benchmark for this RLHF set-up ([Ziegler et al., 2019](https://arxiv.org/abs/1909.08593)). All prompts have the same length, so nanoGPT needs no padding.

**GRPO** (Lecture 6) samples a group of $G$ completions per prompt, scores them, and uses the group statistics as a baseline:
$$A_i=\frac{r_i-\operatorname{mean}(r_{1..G})}{\operatorname{std}(r_{1..G})+\epsilon},\qquad L=-\frac{1}{G\,|o|}\sum_{i,t}\Big(A_i\log\pi_\theta(o_{i,t}\mid q,o_{i,<t})-\beta\,D_{KL,t}\Big).$$
We take one update per batch of samples, so the ratio $\pi_\theta/\pi_{\theta_{old}}$ equals 1 and the clipping in the lecture's objective is inactive; its gradient reduces to the expression above. The KL term (per-token estimator $e^{d}-d-1$, $d=\log\pi_{ref}-\log\pi_\theta$) keeps the policy close to the frozen GPT-2.

For comparison we also run **SFT** on positive IMDB reviews: same prompts format, same 20 completion tokens, same optimiser and learning rate. Every few updates we record the held-out reward and two measures of how much the model changed:
- **weights changed:** the fraction of parameters whose bf16 value differs from GPT-2 (the precision in which models are usually stored);
- **tokens shifted:** the fraction of tokens in held-out human reviews whose log-probability moved by more than 0.5 nats.

In [ ]:
def imdb_rows(split, offset, length=100):
    url = ("https://datasets-server.huggingface.co/rows?dataset=stanfordnlp/imdb&config=plain_text"
           f"&split={split}&offset={offset}&length={length}")
    with urllib.request.urlopen(url) as response:
        return [row["row"] for row in json.load(response)["rows"]]


cache = Path("../data/imdb_lab4.json")
if cache.exists():
    imdb = json.loads(cache.read_text())
else:  # In each split, rows 0-12499 are negative and 12500-24999 positive.
    imdb = {
        "prompts": imdb_rows("train", 0) + imdb_rows("train", 100) + imdb_rows("train", 12500) + imdb_rows("train", 12600),
        "sft": sum((imdb_rows("train", 12700 + 100 * k) for k in range(10)), []),
        "held_out": imdb_rows("test", 0, 30) + imdb_rows("test", 12500, 30),
    }
    cache.parent.mkdir(parents=True, exist_ok=True)
    cache.write_text(json.dumps(imdb))

PROMPT_LEN, NEW_TOKENS = 8, 20
clean = lambda text: " ".join(text.replace("<br />", " ").split())
encode = lambda text: tokenizer.encode(text, max_length=1024, truncation=True)
train_prompts = [encode(clean(r["text"]))[:PROMPT_LEN] for r in imdb["prompts"]]
eval_prompts = [encode(clean(r["text"]))[:PROMPT_LEN] for r in imdb["held_out"]]
sft_examples = [encode(clean(r["text"]))[:PROMPT_LEN + NEW_TOKENS] for r in imdb["sft"]]
held_out_reviews = [encode(clean(r["text"]))[:128] for r in imdb["held_out"]]
print(len(train_prompts), "RL prompts, e.g.", [tokenizer.decode(p) for p in train_prompts[:3]])
print(len(sft_examples), "positive reviews for SFT")

### Exercise 3: Train with GRPO and compare with SFT
**Exercise 3:** Complete `group_advantages`. Check it on the lecture's example: rewards $(1, 1, 0)$ should give advantages $(0.71, 0.71, -1.41)$. Which standard deviation does the lecture use?

**Exercise 3a:** Inspect one group of completions below. Which completions are pushed up and which down? What happens to a group in which all completions receive the same reward?

In [ ]:
P, G = (8, 8) if device.type == "cuda" else (4, 8)  # Prompts per update, completions per prompt.


def group_advantages(rewards, eps=1e-4):
    """rewards: [P, G]. Return group-normalised advantages with the same shape."""
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True, unbiased=False)  # Population std, as in the lecture.
    return (rewards - mean) / (std + eps)


print("Lecture example:", group_advantages(torch.tensor([[1.0, 1.0, 0.0]])))

# Exercise 3a: one group of completions from GPT-2 before training.
torch.manual_seed(0)
prompt = torch.tensor([train_prompts[0]] * G, device=device)
with torch.no_grad():
    sequences = reference.generate(prompt, NEW_TOKENS)
rewards = reward_fn(tokenizer.batch_decode(sequences))
advantages = group_advantages(rewards[None])[0]
print("\nPrompt:", repr(tokenizer.decode(train_prompts[0])))
for seq, r, a in zip(sequences, rewards, advantages):
    print(f"reward {r:.2f}  advantage {a:+.2f}  {tokenizer.decode(seq[PROMPT_LEN:])!r}")

Now train both methods. GRPO stops once the held-out reward reaches `TARGET_REWARD` or after `MAX_STEPS["grpo"]` updates; SFT runs for `MAX_STEPS["sft"]` updates (one SFT update is about four times cheaper than one GRPO update, which also generates text). Reduce the step counts if your computer is slow.

**Exercise 3b:** Which method reaches a higher reward, and after how many updates? Read the middle and right plots at the **same reward**: what fraction of the weights and of the held-out tokens has each method changed? Give one reason why RL on the model's own samples changes so little, and why SFT on human reviews changes so much.

**Exercise 3c:** Read the GRPO samples printed after training. Do you see signs of reward hacking? Which term in the loss limits it, and what would you expect with `BETA = 0` and more updates?

In [ ]:
LR, BETA, TARGET_REWARD = 1e-5, 0.02, 0.8
MAX_STEPS = {"grpo": 60, "sft": 150}
EVAL_EVERY = 10


@torch.no_grad()
def evaluate(model):
    """Held-out reward, fraction of weights changed (bf16), fraction of tokens shifted > 0.5 nats."""
    model.eval()
    prompts = torch.tensor(eval_prompts, device=device).repeat_interleave(2 if device.type == "cuda" else 1, 0)
    reward = reward_fn(tokenizer.batch_decode(model.generate(prompts, NEW_TOKENS))).mean().item()
    changed = sum((p.bfloat16() != q.bfloat16()).sum().item()
                  for p, q in zip(model.parameters(), reference.parameters()))
    total = sum(p.numel() for p in model.parameters())
    shifts = torch.cat([(completion_logprobs(model, x, 1) - completion_logprobs(reference, x, 1)).abs().flatten()
                        for x in (torch.tensor([r], device=device) for r in held_out_reviews)])
    return reward, changed / total, (shifts > 0.5).float().mean().item()


def grpo_step(policy, optimizer):
    prompts = torch.tensor(random.sample(train_prompts, P), device=device).repeat_interleave(G, 0)
    policy.eval()
    with torch.no_grad():
        sequences = policy.generate(prompts, NEW_TOKENS)  # 1) sample G completions per prompt
        rewards = reward_fn(tokenizer.batch_decode(sequences)).view(P, G)  # 2) score them
        ref_logp = completion_logprobs(reference, sequences, PROMPT_LEN)
    advantages = group_advantages(rewards).view(-1, 1)  # 3) group-relative advantages
    policy.train()
    logp = completion_logprobs(policy, sequences, PROMPT_LEN)
    d = ref_logp - logp
    kl = d.exp() - d - 1
    return -(advantages * logp - BETA * kl).mean()  # 4) loss; backpropagated in train()


def sft_step(policy, optimizer):
    batch = torch.tensor(random.sample(sft_examples, P * G), device=device)
    policy.train()
    return -completion_logprobs(policy, batch, PROMPT_LEN).mean()  # Next-token cross-entropy.


def train(method):
    torch.manual_seed(0)
    random.seed(0)
    policy = load_gpt2()
    optimizer = torch.optim.AdamW(policy.parameters(), lr=LR, weight_decay=0.0)
    history = [(0, *evaluate(policy))]
    step_fn = grpo_step if method == "grpo" else sft_step
    for step in range(1, MAX_STEPS[method] + 1):
        loss = step_fn(policy, optimizer)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer.step()
        if step % EVAL_EVERY == 0:
            history.append((step, *evaluate(policy)))
            print(f"{method} step {step:3d}: reward {history[-1][1]:.3f}, "
                  f"weights changed {history[-1][2]:.1%}, tokens shifted {history[-1][3]:.1%}")
            if method == "grpo" and history[-1][1] >= TARGET_REWARD:
                break
    return policy, history


grpo_model, grpo_history = train("grpo")
sft_model, sft_history = train("sft")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for name, history in [("GRPO", grpo_history), ("SFT", sft_history)]:
    steps, reward, weights, tokens = map(list, zip(*history))
    axes[0].plot(steps, reward, marker="o", label=name)
    axes[1].plot([100 * w for w in weights], reward, marker="o", label=name)
    axes[2].plot([100 * t for t in tokens], reward, marker="o", label=name)
axes[0].set(xlabel="Update", ylabel="Held-out reward", title="Performance per update")
axes[1].set(xlabel="Weights changed in bf16 (%)", ylabel="Held-out reward", title="Performance vs weight change")
axes[2].set(xlabel="Held-out tokens shifted > 0.5 nats (%)", ylabel="Held-out reward",
            title="Performance vs behaviour change")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

torch.manual_seed(1)
prompts = torch.tensor(eval_prompts[::12], device=device)
for name, model in [("GPT-2", reference), ("GRPO", grpo_model), ("SFT", sft_model)]:
    model.eval()
    with torch.no_grad():
        texts = tokenizer.batch_decode(model.generate(prompts, NEW_TOKENS))
    print(f"\n=== {name} ===")
    for text, reward in zip(texts, reward_fn(texts).tolist()):
        print(f"{reward:.2f}  {text!r}")

## 5. Visualisation: what did each method change?
For a held-out human review, we compute how much each token's log-probability moved relative to GPT-2: $\Delta_t=\log\pi_\theta(x_t\mid x_{<t})-\log\pi_{ref}(x_t\mid x_{<t})$. Green tokens became more likely; red tokens less likely. Hover over a token to see its exact $\Delta_t$. The first review is negative and the second positive.

### Exercise 4: Read the change maps
**Exercise 4a:** Which kinds of tokens does GRPO change, and in which direction? Relate them to the reward model from Exercise 2.

**Exercise 4b:** Which kinds of tokens does SFT change? Why does training on positive reviews change words such as names, topics, or punctuation?

**Exercise 4c:** Suppose GPT-2 had other skills you wanted to keep (e.g. summarising or answering questions). Based on these maps and the plots in Exercise 3b, which method would you expect to damage those skills more, and why?

In [ ]:
from html import escape
from IPython.display import HTML, display


@torch.no_grad()
def token_deltas(model, ids):
    x = torch.tensor([ids], device=device)
    return (completion_logprobs(model, x, 1) - completion_logprobs(reference, x, 1))[0].cpu()


def change_map(ids, deltas, scale=2.0):
    spans = []
    for token, delta in zip(ids[1:], deltas.tolist()):
        alpha = min(abs(delta) / scale, 1.0)
        rgb = "22,163,74" if delta > 0 else "220,38,38"
        spans.append(f'<span title="{delta:+.2f}" style="background:rgba({rgb},{alpha:.2f});'
                     f'color:inherit;border-radius:3px">{escape(tokenizer.decode([token]))}</span>')
    return "".join(spans)


grpo_model.eval()
sft_model.eval()
rows = []
for ids in [held_out_reviews[3][:70], held_out_reviews[33][:70]]:
    for name, model in [("GRPO", grpo_model), ("SFT", sft_model)]:
        deltas = token_deltas(model, ids)
        rows.append(f'<tr><td style="padding:6px 10px;vertical-align:top;font-weight:600">{name}<br>'
                    f'<span style="font-weight:400;font-size:85%">{(deltas.abs() > 0.5).float().mean():.0%} '
                    f'of tokens<br>moved &gt; 0.5</span></td><td style="padding:6px 10px;line-height:1.7">'
                    f'{change_map(ids, deltas)}</td></tr>')
display(HTML('<table style="border-collapse:collapse;font-family:monospace;font-size:13px">'
             + "".join(rows) + "</table>"))

# Which tokens did each method push up most, across all held-out reviews?
for name, model in [("GRPO", grpo_model), ("SFT", sft_model)]:
    total, count = {}, {}
    for ids in held_out_reviews:
        for token, delta in zip(ids[1:], token_deltas(model, ids).tolist()):
            total[token] = total.get(token, 0.0) + delta
            count[token] = count.get(token, 0) + 1
    mean = {t: total[t] / count[t] for t in total if count[t] >= 3}
    ranked = sorted(mean, key=mean.get)
    fmt = lambda tokens: ", ".join(f"{tokenizer.decode([t]).strip()!r} ({mean[t]:+.1f})" for t in tokens)
    print(f"{name}: most increased: {fmt(ranked[::-1][:8])}")
    print(f"{' ' * len(name)}  most decreased: {fmt(ranked[:8])}")